# Phase 5 — Architecture 4: LegalBlock-GQA

In [1]:
%pip -q install "tokenizers==0.21.4" "huggingface_hub==0.34.4" cloudpickle scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.34.4 which is incompatible.
transformers 5.16.1 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.21.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import shutil

SOURCE = Path('/kaggle/input/datasets/harshvardhanvar/a4-p5-resume/phase5_a4')
OUT = Path('/kaggle/working/phase5_a4')

shutil.copytree(SOURCE, OUT, dirs_exist_ok=True)

PosixPath('/kaggle/working/phase5_a4')

In [3]:
# Architecture 4: LegalBlock-GQA — RoPE, RMSNorm, SwiGLU, grouped-query attention
import os, sys, json, math, time, random, gc, zipfile, subprocess, cloudpickle, hashlib, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, accuracy_score, classification_report

BASE = Path('/kaggle/input/datasets/harshvardhanvar/a4-p3')
OUT = Path('/kaggle/working/phase5_a4')
HF_TOKEN = os.environ['HF_TOKEN']
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '2'
OUT.mkdir(parents=True, exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
C = json.loads((BASE/'config.json').read_text())
R = dict(base=str(BASE), out=str(OUT), seed=781, length=512, stride=384,
         chunks=8, chunk_batch=2, accum=8, rank=16, eval_steps=200, patience=5)
METHODS = {
    'full': dict(epochs=6, lr=2e-5),
    'partial': dict(epochs=8, lr=5e-5),
    'lora': dict(epochs=10, lr=2e-4),
    'lorma': dict(epochs=10, lr=2e-4),
    'ia3': dict(epochs=10, lr=5e-4),
    'distill': dict(epochs=8, lr=4e-5),
}
random.seed(R['seed'])
np.random.seed(R['seed'])
torch.manual_seed(R['seed'])


## Data

In [3]:
revision = HfApi(token=HF_TOKEN).dataset_info(REPO).sha
files = ['Train/train.json', 'Train/dev.json', 'Test/test.json']
DOCS = {}
for name in files:
    path = hf_hub_download(REPO, 'Phase_5_LSI/'+name, repo_type='dataset',
                           revision=revision, token=HF_TOKEN)
    raw = json.loads(Path(path).read_text())
    if isinstance(raw, dict):
        raw = [raw] if 'id' in raw and 'text' in raw else [dict(v, id=k) for k,v in raw.items()]
    DOCS[Path(name).stem] = raw
labels = sorted({int(y) for split in ['train','dev'] for d in DOCS[split] for y in d['labels']})
label2id = {y:i for i,y in enumerate(labels)}
R['classes'] = len(labels)
tok = Tokenizer.from_file(str(BASE/'tokenizer.json'))
R['pad'] = tok.token_to_id('<pad>')
R['bos'] = tok.token_to_id('<bos>')

def encode_docs(docs, labelled):
    rows = []
    for doc in docs:
        ids = tok.encode(doc['text'], add_special_tokens=False).ids
        starts = list(range(0, max(1,len(ids)-R['length']+1), R['stride']))
        tail = max(0,len(ids)-R['length'])
        if starts[-1] != tail: starts.append(tail)
        if len(starts)>R['chunks']:
            starts = [starts[i] for i in np.linspace(0,len(starts)-1,R['chunks'],dtype=int)]
        chunks = [ids[s:s+R['length']] or [R['pad']] for s in starts]
        y = np.zeros(len(labels),dtype=np.float32)
        if labelled:
            for label in doc['labels']: y[label2id[int(label)]] = 1
        rows.append(dict(id=doc['id'], chunks=chunks, y=y))
    return rows

DATA = {s:encode_docs(ds,s!='test') for s,ds in DOCS.items()}
R['counts'] = np.stack([v['y'] for v in DATA['train']]).sum(0).tolist()
with (OUT/'data.pkl').open('wb') as f:
    cloudpickle.dump(dict(rows=DATA,labels=labels),f)
R['run_id'] = hashlib.sha256(json.dumps([C,R,METHODS,labels,revision,
    hashlib.sha256((BASE/'tokenizer.json').read_bytes()).hexdigest()],sort_keys=True).encode()).hexdigest()
manifest = dict(architecture=4, config=C, settings=R, methods=METHODS, labels=labels,
                repo=REPO, revision=revision, files=files)
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'tokenizer.json',OUT/'tokenizer.json')
print(' | '.join(f'{s}: {len(v):,} documents' for s,v in DATA.items())+f' | {len(labels)} statutes')


Phase_5_LSI/Train/train.json:   0%|          | 0.00/299M [00:00<?, ?B/s]

Phase_5_LSI/Train/dev.json:   0%|          | 0.00/79.4M [00:00<?, ?B/s]

Phase_5_LSI/Test/test.json:   0%|          | 0.00/98.1M [00:00<?, ?B/s]

train: 42,750 documents | dev: 10,181 documents | test: 13,019 documents | 100 statutes


## Architecture 4: LegalBlock-GQA backbone

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c, global_attn):
        super().__init__()
        self.h, self.kv, self.d = c['heads'], c['kv_heads'], c['dim'] // c['heads']
        self.window, self.global_attn = c['window'], global_attn
        self.qkv = nn.Linear(c['dim'], (self.h + 2*self.kv)*self.d, bias=False)
        self.out = nn.Linear(c['dim'], c['dim'], bias=False)
        self.register_buffer('freq', 1 / (10000 ** (torch.arange(0, self.d, 2).float()/self.d)), persistent=False)
    def rope(self, x, pos):
        angle = (torch.arange(pos, pos+x.size(2), device=x.device).float()[:, None] * self.freq).to(x.dtype)
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), -1).flatten(-2)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).split([self.h*self.d, self.kv*self.d, self.kv*self.d], -1)
        q = self.rope(q.view(b,t,self.h,self.d).transpose(1,2), pos)
        k = self.rope(k.view(b,t,self.kv,self.d).transpose(1,2), pos)
        v = v.view(b,t,self.kv,self.d).transpose(1,2)
        if cache is not None and (self.global_attn or pos % self.window != 0):
            k, v = torch.cat((cache[0], k), 2), torch.cat((cache[1], v), 2)
        saved = None
        if use_cache:
            start = 0 if self.global_attn else ((pos+t-1)//self.window)*self.window - pos
            start = max(0, start) if cache is None else 0
            saved = (k[:,:,start:], v[:,:,start:])
        k, v = k.repeat_interleave(self.h//self.kv, 1), v.repeat_interleave(self.h//self.kv, 1)
        if t == 1:
            y = F.scaled_dot_product_attention(q, k, v)
        elif self.global_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            w = self.window
            pad = (-t) % w
            q, k, v = [F.pad(z, (0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                       .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
            y = y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1)), saved

class Block(nn.Module):
    def __init__(self, c, i):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c['dim']), RMSNorm(c['dim'])
        self.attn = Attention(c, (i+1) % c['global_every'] == 0)
        self.up = nn.Linear(c['dim'], 2*c['ff'], bias=False)
        self.down = nn.Linear(c['ff'], c['dim'], bias=False)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        y, cache = self.attn(self.n1(x), cache, pos, use_cache)
        x = x + y
        a, b = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(a)*b), cache

class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c['vocab'], c['dim'])
        self.blocks = nn.ModuleList([Block(c, i) for i in range(c['layers'])])
        self.norm = RMSNorm(c['dim'])
        self.apply(self.init_weights)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=.02/math.sqrt(2*c['layers']))
            nn.init.normal_(block.down.weight, std=.02/math.sqrt(2*c['layers']))
    def init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=.02)


In [5]:
def features_for_chunks(lm,x,sizes,r):
    h = lm.emb(x)
    for block in lm.blocks:
        h,_ = checkpoint(block,h,use_reentrant=False) if lm.training else block(h)
    h = lm.norm(h)
    mask = torch.arange(x.size(1),device=x.device)[None]<sizes[:,None]
    mean = (h*mask[:,:,None]).sum(1)/sizes[:,None]
    last = h[torch.arange(len(x),device=x.device),sizes-1]
    return torch.cat((mean,last),-1)


## Six methods

In [6]:
class LowRankAdapter(nn.Module):
    def __init__(self, base, rank, multiplicative=False):
        super().__init__()
        self.base = base
        self.multiplicative = multiplicative
        width = base.out_features if multiplicative else base.in_features
        self.a = nn.Parameter(torch.empty(rank,width))
        self.b = nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.a,a=math.sqrt(5))
        self.dropout = nn.Dropout(.05)
        self.scale = 2.
    def forward(self,x):
        y = self.base(x)
        h = self.dropout(x)
        if self.multiplicative:
            h = F.linear(h,self.base.weight)
        return y+self.scale*F.linear(F.linear(h,self.a),self.b)

class ActivationScale(nn.Module):
    def __init__(self,base,on_input=False):
        super().__init__()
        self.base,self.on_input = base,on_input
        self.scale = nn.Parameter(torch.ones(base.in_features if on_input else base.out_features))
    def forward(self,x):
        if self.on_input: return self.base(x*self.scale.to(x.dtype))
        y = self.base(x)
        return y*self.scale.to(y.dtype)

class SplitProjection(nn.Module):
    def __init__(self, base, sizes, names):
        super().__init__()
        self.names = names
        for name,w in zip(names,base.weight.split(sizes,0)):
            layer = nn.Linear(base.in_features,w.size(0),bias=False)
            with torch.no_grad(): layer.weight.copy_(w)
            setattr(self,name,layer)
    def forward(self,x):
        return torch.cat([getattr(self,name)(x) for name in self.names],-1)

class StatuteModel(nn.Module):
    def __init__(self,lm,r):
        super().__init__()
        self.lm, self.r = lm,r
        self.project = nn.Sequential(nn.Linear(2*lm.c['dim'],256),nn.Tanh(),nn.Dropout(.15))
        self.evidence = nn.Linear(256,r['classes'])
        self.attention = nn.Linear(256,r['classes'],bias=False)
        nn.init.zeros_(self.attention.weight)
    def forward(self,chunks):
        features = []
        device = next(self.parameters()).device
        for at in range(0,len(chunks),self.r['chunk_batch']):
            group = chunks[at:at+self.r['chunk_batch']]
            length = math.ceil(max(map(len,group))/8)*8
            x = torch.full((len(group),length),self.r['pad'],dtype=torch.long,device=device)
            sizes = torch.tensor([len(v) for v in group],device=device)
            for i,ids in enumerate(group): x[i,:len(ids)] = torch.tensor(ids,device=device)
            features.append(features_for_chunks(self.lm,x,sizes,self.r))
        h = self.project(torch.cat(features,0))
        weights = self.attention(h).float().softmax(0)
        return (weights*self.evidence(h).float()).sum(0)

def make_model(method,c,r):
    saved = torch.load(Path(r['base'])/'best.pt',map_location='cpu',weights_only=False)
    config = saved.get('config',c).copy()
    lm = LegalLM(config)
    lm.load_state_dict(saved['model'],strict=True)
    del saved
    if method=='distill':
        chosen = np.linspace(0,len(lm.blocks)-1,8,dtype=int)
        lm.blocks = nn.ModuleList([lm.blocks[i] for i in chosen])
    d = config['dim']
    kv = config['kv_heads']*(d//config['heads'])
    for b in lm.blocks:
        b.attn.qkv = SplitProjection(b.attn.qkv,[d,kv,kv],['q_proj','k_proj','v_proj'])
    if method not in ['full','distill']: lm.requires_grad_(False)
    if method=='partial':
        for b in lm.blocks[-6:]: b.requires_grad_(True)
        lm.norm.requires_grad_(True)
    for name,layer in list(lm.named_modules()):
        if not isinstance(layer,nn.Linear): continue
        key = name.split('.')[-1]
        adapted = None
        if method in ['lora','lorma'] and key in ['q_proj','v_proj','out','up','down']:
            adapted = LowRankAdapter(layer,r['rank'],multiplicative=method=='lorma')
        if method=='ia3' and key in ['k_proj','v_proj','down']:
            adapted = ActivationScale(layer,on_input=key=='down')
        if adapted is not None:
            parent,key = name.rsplit('.',1)
            setattr(lm.get_submodule(parent),key,adapted)
    model = StatuteModel(lm,r)
    return model


## Predictions and calibration

In [7]:
@torch.inference_mode()
def predict(model,rows,device,r):
    model.eval()
    logits = np.empty((len(rows),r['classes']),dtype=np.float32)
    for i,row in enumerate(rows):
        with torch.autocast('cuda',dtype=torch.float16):
            logits[i] = model(row['chunks']).float().cpu().numpy()
    return logits

def probabilities(z):
    return 1/(1+np.exp(-np.clip(z,-40,40)))

def decode(p,threshold):
    pred = p>=threshold
    empty = ~pred.any(1)
    pred[np.flatnonzero(empty),p[empty].argmax(1)] = True
    return pred

def calibrate(p,gold):
    grid = np.arange(.1,.81,.05)
    scores = [f1_score(gold,decode(p,t),average='macro',zero_division=0) for t in grid]
    global_t = float(grid[int(np.argmax(scores))])
    thresholds = np.full(p.shape[1],global_t)
    for j in range(p.shape[1]):
        positives = int(gold[:,j].sum())
        if positives>=5 and positives<len(gold):
            values = [f1_score(gold[:,j],p[:,j]>=t,zero_division=0) for t in grid]
            local_t = float(grid[int(np.argmax(values))])
            weight = positives/(positives+20.)
            thresholds[j] = weight*local_t+(1-weight)*global_t
    score = f1_score(gold,decode(p,thresholds),average='macro',zero_division=0)
    if score<scores[int(np.argmax(scores))]:
        thresholds[:] = global_t
        score = max(scores)
    return thresholds,float(score)

def write_submission(rows,labels,p,threshold,path):
    chosen = decode(p,threshold)
    prediction = [dict(id=row['id'],pred=[int(labels[j]) for j in np.flatnonzero(y)])
                  for row,y in zip(rows,chosen)]
    payload = json.dumps(prediction,ensure_ascii=False,indent=2)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',payload)
    return hashlib.sha256(payload.encode()).hexdigest()


## Training

In [8]:
def run_method(gpu,method,c,r,methods,teacher=None):
    torch.set_num_threads(2)
    torch.cuda.set_device(gpu)
    device = torch.device('cuda',gpu)
    torch.manual_seed(r['seed'])
    np.random.seed(r['seed'])
    random.seed(r['seed'])
    root = Path(r['out'])
    folder = root/method
    folder.mkdir(exist_ok=True)
    with (root/'data.pkl').open('rb') as f: data = cloudpickle.load(f)
    train,dev,test = [data['rows'][s] for s in ['train','dev','test']]
    gold = np.stack([v['y'] for v in dev])
    soft = None
    if method=='distill':
        tm = make_model(teacher,c,r)
        ck = torch.load(root/teacher/'best.pt',map_location='cpu',weights_only=False)
        tm.load_state_dict(ck['weights'],strict=False)
        tm.to(device)
        soft = predict(tm,train,device,r)
        np.save(folder/'teacher_logits.npy',soft)
        del tm,ck
        gc.collect()
        torch.cuda.empty_cache()
    model = make_model(method,c,r).to(device)
    params = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
    body = [p for n,p in params if n.startswith('lm.')]
    head = [p for n,p in params if not n.startswith('lm.')]
    lr = methods[method]['lr']
    opt = torch.optim.AdamW([dict(params=body,lr=lr,initial_lr=lr),
                             dict(params=head,lr=3e-4,initial_lr=3e-4)],weight_decay=.01)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024)
    count = np.array(r['counts'])
    weight = np.sqrt((len(train)-count)/np.maximum(count,1)).clip(1,8)
    weight = torch.tensor(weight,dtype=torch.float32,device=device)
    nsteps = math.ceil(len(train)/r['accum'])
    total = methods[method]['epochs']*nsteps
    warmup = max(1,int(.06*total))
    best,stale,step,history = -1.,0,0,[]
    start = time.monotonic()
    total_params = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for _,p in params)
    torch.cuda.reset_peak_memory_stats(device)
    print(f'{method} | GPU {gpu} | {total_params:,} parameters | {trained:,} trainable',flush=True)
    stop = False
    for epoch in range(1,methods[method]['epochs']+1):
        order = np.random.default_rng(r['seed']+epoch).permutation(len(train))
        loss_sum,seen = 0.,0
        for at in range(0,len(order),r['accum']):
            model.train()
            opt.zero_grad(set_to_none=True)
            group = order[at:at+r['accum']]
            progress = max(0.,(step-warmup)/max(1,total-warmup))
            factor = min(1.,(step+1)/warmup)*(.1+.45*(1+math.cos(math.pi*progress)))
            for g in opt.param_groups: g['lr'] = g['initial_lr']*factor
            for i in group:
                y = torch.tensor(train[i]['y'],device=device)
                with torch.autocast('cuda',dtype=torch.float16):
                    z = model(train[i]['chunks'])
                    hard = F.binary_cross_entropy_with_logits(z.float(),y,pos_weight=weight)
                    loss = hard
                    if soft is not None:
                        target = torch.tensor(soft[i],device=device)
                        kd = F.binary_cross_entropy_with_logits(z.float()/2,(target/2).sigmoid())*4
                        loss = .5*hard+.5*kd
                scaler.scale(loss/len(group)).backward()
                loss_sum += loss.detach().item()
                seen += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for _,p in params],1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step%r['eval_steps']==0 or at+len(group)>=len(order):
                logits = predict(model,dev,device,r)
                threshold,score = calibrate(probabilities(logits),gold)
                improved = score>best
                stale = 0 if improved else stale+1
                if improved:
                    best = score
                    weights = {n:p.detach().cpu().float() for n,p in params}
                    torch.save(dict(weights=weights,method=method,epoch=epoch,step=step,
                                    score=best,threshold=threshold.tolist(),teacher=teacher),folder/'best.tmp')
                    os.replace(folder/'best.tmp',folder/'best.pt')
                    del weights
                    np.save(folder/'dev_logits.npy',logits)
                row = dict(epoch=epoch,step=step,loss=loss_sum/seen,macro_f1=score,stale=stale)
                history.append(row)
                (folder/'history.json').write_text(json.dumps(history,indent=2))
                print(f'{method} | epoch {epoch} step {step} | loss {row["loss"]:.4f} | dev F1 {score:.4f} | stale {stale}/5',flush=True)
                if stale>=r['patience']:
                    stop = True
                    break
        if stop: break
    ck = torch.load(folder/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(ck['weights'],strict=False)
    threshold = np.array(ck['threshold'])
    test_logits = predict(model,test,device,r)
    np.save(folder/'test_logits.npy',test_logits)
    sub = root/'submissions'
    sub.mkdir(exist_ok=True)
    write_submission(test,data['labels'],probabilities(test_logits),threshold,sub/f'submission_A4_{method}.zip')
    pred = decode(probabilities(np.load(folder/'dev_logits.npy')),threshold)
    report = classification_report(gold,pred,target_names=[str(v) for v in data['labels']],
                                   zero_division=0,output_dict=True)
    (folder/'classification_report.json').write_text(json.dumps(report,indent=2))
    result = dict(method=method,run_id=r['run_id'],macro_f1=best,
                  micro_f1=float(f1_score(gold,pred,average='micro',zero_division=0)),
                  subset_accuracy=float(accuracy_score(gold,pred)),parameters=total_params,trainable=trained,
                  epoch=ck['epoch'],step=ck['step'],last_step=step,teacher=teacher,
                  stopping_reason='patience' if stop else 'epoch_limit',threshold=threshold.tolist(),
                  minutes=(time.monotonic()-start)/60,peak_gb=torch.cuda.max_memory_allocated(device)/2**30)
    (folder/'done.json').write_text(json.dumps(result,indent=2))


## Two-GPU training

In [9]:
def run_jobs(names, worker=run_method):
    pending, running, finished = list(names), {}, set()
    first = ['full','partial','lora','lorma','ia3']
    while pending or running:
        for gpu,(process,path,method) in list(running.items()):
            status = process.poll()
            if status is not None:
                if status != 0:
                    raise subprocess.CalledProcessError(status,process.args)
                path.unlink()
                del running[gpu]
                finished.add(method)
                print(f'GPU {gpu} finished {method}',flush=True)
        for gpu in range(2):
            if gpu in running:
                continue
            ready = [m for m in pending if m!='distill' or set(first).issubset(finished)]
            if not ready:
                continue
            method = ready[0]
            teacher = None
            if method=='distill':
                teacher = max(first,key=lambda m:json.loads((OUT/m/'done.json').read_text())['macro_f1'])
            path = OUT/f'worker_{gpu}.pkl'
            with path.open('wb') as f:
                cloudpickle.dump((worker,(gpu,method,C,R,METHODS,teacher)),f)
            command = "import cloudpickle,sys; f=open(sys.argv[1],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(*args)"
            process = subprocess.Popen([sys.executable,'-u','-c',command,str(path)],env=os.environ.copy())
            running[gpu] = (process,path,method)
            pending.remove(method)
            print(f'GPU {gpu} starting {method} | PID {process.pid}',flush=True)
        if pending and not running:
            raise RuntimeError('Distillation requires the first five methods in this job queue')
        if running:
            time.sleep(.5)

run_jobs(list(METHODS))


GPU 0 starting full | PID 95


GPU 1 starting partial | PID 96


full | GPU 0 | 262,078,756 parameters | 262,078,756 trainable
partial | GPU 1 | 262,078,756 parameters | 58,063,396 trainable


partial | epoch 1 step 200 | loss 0.6732 | dev F1 0.0811 | stale 0/5


full | epoch 1 step 200 | loss 0.6479 | dev F1 0.0786 | stale 0/5


partial | epoch 1 step 400 | loss 0.5674 | dev F1 0.0802 | stale 1/5


... (51 outputs left out)

partial | epoch 2 step 6000 | loss 0.2739 | dev F1 0.3395 | stale 0/5


full | epoch 1 step 5000 | loss 0.3374 | dev F1 0.3383 | stale 0/5


partial | epoch 2 step 6200 | loss 0.2730 | dev F1 0.3382 | stale 1/5


full | epoch 1 step 5200 | loss 0.3354 | dev F1 0.3367 | stale 1/5


## Submissions

In [ ]:
gold = np.stack([v['y'] for v in DATA['dev']])
results = [json.loads((OUT/m/'done.json').read_text()) for m in METHODS]
ranked = sorted(results,key=lambda v:v['macro_f1'],reverse=True)
members = [v['method'] for v in ranked[:3]]
dev_probs = np.stack([probabilities(np.load(OUT/m/'dev_logits.npy')) for m in members])
test_probs = np.stack([probabilities(np.load(OUT/m/'test_logits.npy')) for m in members])
candidates = [np.ones(3)/3,np.array([max(v['macro_f1'],1e-6) for v in ranked[:3]])]
candidates[1] /= candidates[1].sum()
trials = [calibrate(np.einsum('m,mnc->nc',w,dev_probs),gold) for w in candidates]
best = int(np.argmax([v[1] for v in trials]))
threshold,score = trials[best]
w = candidates[best]
ensemble = np.einsum('m,mnc->nc',w,test_probs)
selection = dict(members=members,weights=w.tolist(),threshold=threshold.tolist(),dev_macro_f1=score)
(OUT/'ensemble.json').write_text(json.dumps(selection,indent=2))
results.append(dict(method='ensemble',macro_f1=score,members=members))
SUB = OUT/'submissions'
write_submission(DATA['test'],labels,ensemble,threshold,SUB/'submission_A4_ensemble.zip')
(OUT/'results.json').write_text(json.dumps(results,indent=2))
for v in sorted(results,key=lambda v:v['macro_f1'],reverse=True):
    print(f'{v["method"]:10s} dev Macro-F1 {v["macro_f1"]:.4f}')
with zipfile.ZipFile(OUT/'all_submissions_A4.zip','w',zipfile.ZIP_DEFLATED) as z:
    for method in list(METHODS)+['ensemble']:
        p = SUB/f'submission_A4_{method}.zip'
        z.write(p,p.name)
from IPython.display import FileLink, display
display(FileLink(str(OUT/'all_submissions_A4.zip')))


In [12]:
from IPython.display import FileLink, display

SAVED = Path('/kaggle/working/phase5_a4')
SUB = Path('/kaggle/working/submissions')
SUB.mkdir(exist_ok=True)

manifest = json.loads((SAVED/'manifest.json').read_text())
C, R = manifest['config'], manifest['settings']
R['base'] = str(BASE)
with (SAVED/'data.pkl').open('rb') as f:
    data = cloudpickle.load(f)

for method in ['full', 'partial']:
    ck = torch.load(SAVED/method/'best.pt',
                    map_location='cpu', weights_only=False)
    model = make_model(method, C, R)
    model.load_state_dict(ck['weights'], strict=False)
    model.to('cuda:0')

    logits = predict(model, data['rows']['test'], 'cuda:0', R)
    path = SUB/f'submission_A4_{method}.zip'
    write_submission(data['rows']['test'], data['labels'],
                     probabilities(logits), np.array(ck['threshold']), path)
    print(f"{method}: saved dev F1 {ck['score']:.4f}")
    display(FileLink(str(path)))

    del model, ck
    gc.collect()
    torch.cuda.empty_cache()

full: saved dev F1 0.3383


/kaggle/working/submissions/submission_A4_full.zip

partial: saved dev F1 0.3395


/kaggle/working/submissions/submission_A4_partial.zip

In [9]:
from IPython.display import FileLink, display

SAVED = Path('/kaggle/working/phase5_a4')
OUT = Path('/kaggle/working/phase5_a4_distill')
OUT.mkdir(parents=True, exist_ok=True)

manifest = json.loads((SAVED/'manifest.json').read_text())
C, R = manifest['config'], manifest['settings']
R.update(base=str(BASE), out=str(OUT), patience=5)
METHODS = {'distill': dict(epochs=8, lr=4e-5)}

(OUT/'full').mkdir(exist_ok=True)
shutil.copyfile(SAVED/'full'/'best.pt', OUT/'full'/'best.pt')
shutil.copyfile(SAVED/'data.pkl', OUT/'data.pkl')
manifest.update(config=C, settings=R, methods=METHODS)
(OUT/'manifest.json').write_text(json.dumps(manifest, indent=2))

run_method(0, 'distill', C, R, METHODS, teacher='full')
gc.collect()
torch.cuda.empty_cache()

distill | GPU 0 | 108,693,796 parameters | 108,693,796 trainable
distill | epoch 1 step 200 | loss 1.5648 | dev F1 0.0720 | stale 0/5
distill | epoch 1 step 400 | loss 1.4286 | dev F1 0.0652 | stale 1/5
distill | epoch 1 step 600 | loss 1.3668 | dev F1 0.0563 | stale 2/5
distill | epoch 1 step 800 | loss 1.3345 | dev F1 0.0983 | stale 0/5
distill | epoch 1 step 1000 | loss 1.3105 | dev F1 0.1373 | stale 0/5
distill | epoch 1 step 1200 | loss 1.2886 | dev F1 0.2202 | stale 0/5
distill | epoch 1 step 1400 | loss 1.2684 | dev F1 0.2479 | stale 0/5
distill | epoch 1 step 1600 | loss 1.2515 | dev F1 0.2629 | stale 0/5
distill | epoch 1 step 1800 | loss 1.2359 | dev F1 0.2741 | stale 0/5
distill | epoch 1 step 2000 | loss 1.2232 | dev F1 0.2829 | stale 0/5
distill | epoch 1 step 2200 | loss 1.2132 | dev F1 0.2881 | stale 0/5
distill | epoch 1 step 2400 | loss 1.2044 | dev F1 0.2861 | stale 1/5
distill | epoch 1 step 2600 | loss 1.1968 | dev F1 0.2951 | stale 0/5
distill | epoch 1 step 2800 |

In [11]:
from IPython.display import FileLink, display

ck = torch.load(OUT/'distill'/'best.pt',
                map_location='cpu', weights_only=False)
model = make_model('distill', C, R)
model.load_state_dict(ck['weights'], strict=False)
model.to('cuda:0')

with (OUT/'data.pkl').open('rb') as f:
    data = cloudpickle.load(f)

logits = predict(model, data['rows']['test'], 'cuda:0', R)
path = OUT/'submission_A4_distill.zip'
write_submission(data['rows']['test'], data['labels'],
                 probabilities(logits), np.array(ck['threshold']), path)

print(f"Student: epoch {ck['epoch']} | dev F1 {ck['score']:.4f}")
display(FileLink(str(path)))

del model, ck
gc.collect()
torch.cuda.empty_cache()

Student: epoch 2 | dev F1 0.3293


/kaggle/working/phase5_a4_distill/submission_A4_distill.zip

## Results